In [1]:
import pandas as pd
from transformers import CLIPTokenizer, CLIPTextModel
import numpy as np
import pickle
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio, random
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity

"Pierre-Auguste Renoir"
"Käthe Kollwitz"
"Max Ernst"
"Karel Appel"


In [2]:
Artist_name ="Max Ernst"

In [3]:
claude_label = pd.read_excel(f"LLM Request\\comment_annotation_claude_{Artist_name.split(" ")[-1]}.xlsx")

In [4]:
claude_label.shape

(1000, 10)

In [5]:
sum(claude_label['artwork id'].duplicated())

0

In [6]:
gemini_label = pd.read_excel(f"LLM Request\\comment_annotation_gemini_{Artist_name.split(" ")[-1]}.xlsx")

In [7]:
gemini_label.shape

(1000, 10)

In [8]:
sum(gemini_label['artwork id'].duplicated())

0

In [9]:
openai_label = pd.read_excel(f"LLM Request\\comment_annotation_openai_{Artist_name.split(" ")[-1]}.xlsx")

In [10]:
openai_label.shape

(1000, 10)

In [11]:
sum(openai_label['artwork id'].duplicated())

0

In [12]:
full_df = claude_label.merge(openai_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=("_claude","_openai"))

In [13]:
full_df.shape

(1000, 14)

In [14]:
full_df

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer_claude,artistic_value_comment_claude,creativity_answer_claude,creativity_comment_claude,artistic_value_answer_openai,artistic_value_comment_openai,creativity_answer_openai,creativity_comment_openai
0,287,La Forêt,Max,Ernst,1925,German,High,Max Ernst was one of the most materially inven...,Yes,Ernst invented the frottage technique at Porni...,High,"Max Ernst's ""La Forêt"" (The Forest), painted a...",Yes,"""La Forêt"" exemplifies Ernst's creativity thro..."
1,875,Tremblement de terre printanier or Trois tremb...,Max,Ernst,1964,German,High,This exceptional painting from Ernst's mature ...,Yes,The work invites interpretation as expressing ...,High,"Max Ernst's 1964 painting, ""Tremblement de ter...",Yes,"""Tremblement de terre printanier"" exemplifies ..."
2,2196,Dormeuse,Max,Ernst,1955,German,"authoritative commentary on Max Ernst's ""Dorme...",What I cannot provide:**\nWithout specific sch...,"for authoritative commentary on Max Ernst's ""D...",What I cannot provide:**\nWithout specific sch...,High,"Max Ernst's ""Dormeuse"" (1955) exemplifies his ...",Yes,"""Dormeuse"" embodies Ernst's commitment to arti..."
3,3062,Le chant de la grenouille,Max,Ernst,1957,German,authoritative commentary on this Max Ernst art...,I cannot provide the analysis you requested** ...,for authoritative commentary on this Max Ernst...,I cannot provide the analysis you requested** ...,High,"Max Ernst's ""Le chant de la grenouille"" (1957)...",Yes,"""Le chant de la grenouille"" demonstrates Max E..."
4,3491,Drapeau,Max,Ernst,1967,German,"scholarly commentary on Max Ernst's ""Drapeau"" ...",Without authoritative source material on this ...,"for scholarly commentary on Max Ernst's ""Drape...",Without authoritative source material on this ...,High,"Max Ernst's 1967 sculpture ""Grand Grenouille"" ...",Yes,"""Grand Grenouille"" embodies Ernst's creative i..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,428142056,Elfo III,Max,Ernst,1966,German,Moderate to High,"""Elfo III"" is a glass sculpture created in 196...",Conditional,The available sources do not provide sufficien...,High,"Max Ernst's ""Elfo III,"" created in 1966, exemp...",Yes,"Ernst's ""Elfo III"" embodies a significant crea..."
996,428143973,La ballade du soldat (bk by Georges Ribemont D...,Max,Ernst,1972,German,High,This 1972 illustrated book with lithographs fo...,Limited,While Ernst's contributions to artistic innova...,High,"""La Ballade du Soldat"" is a significant collab...",Yes,"Max Ernst's illustrations for ""La Ballade du S..."
997,428154983,Au liège rendu par la mer,Max,Ernst,1969,German,scholarly commentary on this Max Ernst artwork...,Without access to authoritative critical sourc...,for scholarly commentary on this Max Ernst art...,Without access to authoritative critical sourc...,High,"Max Ernst's 1969 painting, 'Au liège rendu par...",Yes,'Au liège rendu par la mer' demonstrates Ernst...
998,428154984,Endlose Spiele bereiten sich vor,Max,Ernst,1972,German,NaN,NaN,NaN,NaN,High,"Max Ernst's 1972 serigraph ""Endlose Spiele ber...",Yes,"""Endlose Spiele bereiten sich vor"" showcases E..."


In [15]:
full_df = full_df.merge(openai_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=(None,"_openai"))

In [16]:
full_df.shape

(1000, 18)

# Embedding Convert

In [17]:
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = text_model.to(device)

In [19]:
def convert_one_row(model,i,texts,device):
    try:
        inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        text_embeds = outputs.pooler_output
        text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
        text_embeds=text_embeds.cpu().numpy()
    except Exception as e:
        print(f"Error processing {i}: {e}")
        text_embeds = np.zeros([2,512])
    return i, text_embeds

In [28]:
dataset = "claude"
if dataset =="claude":
    df = claude_label.copy()
elif dataset =="gemini":
    df = gemini_label.copy()
elif dataset =="openai":
    df = openai_label.copy()

In [29]:
number_size=df.shape[0]
#number_size=10
# range_start = 30000
range_start = 0
range_end = min(range_start+number_size,df.shape[0])
N = min(number_size, df.shape[0]-range_start)

In [30]:
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
embeddings = np.zeros(N, dtype=object)
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = {
        ex.submit(convert_one_row, model,i, 
                  [df.iloc[i].artistic_value_comment,df.iloc[i].creativity_comment],
                  device): i
        for i in range(range_start,range_end)
    }
    for fut in as_completed(futures):
        i, text_embeds = fut.result()
        embeddings[i-range_start] =text_embeds
        if i % 1000 == 0:
            print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2025-12-16 04:45:42: Start
Error processing 9: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
Error processing 33: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
2025-12-16 04:45:42: 0
Error processing 49: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
Error processing 193: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
Error processing 227: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
Error processing 272: text input must be of type `str` (sin

In [31]:
np.save(f"clip_embeddings_{dataset}_{Artist_name.split(" ")[-1]}.npy", embeddings)

# Comment Check Consistency Rough

In [152]:
consistent_creative=[]
consistent_artist=[]
consistent_overall=[]
for i in range(full_df.shape[0]):
    row = full_df.iloc[i]
    if (row.artistic_value_answer_claude == row.artistic_value_answer.strip()) & (row.artistic_value_answer.strip() == row.artistic_value_answer_gemini.strip()):
        artist_con=1
        consistent_artist.append(1)
    else:
        artist_con=0
        consistent_artist.append(0)
    if (row.creativity_answer_claude == row.creativity_answer.strip()) & (row.creativity_answer.strip() == row.creativity_answer_gemini.strip()):
        creative=1
        consistent_creative.append(1)
    else:
        creative=0
        consistent_creative.append(0)

    if artist_con+creative==2:
        consistent_overall.append(1)
    else:
        consistent_overall.append(0)

In [164]:
np.sum(consistent_artist)

np.int64(78)

In [163]:
np.sum(consistent_creative)

np.int64(73)

In [153]:
np.sum(consistent_overall)

np.int64(73)

In [154]:
check=full_df.copy()
check["creative_consist"]=consistent_creative
check["artistic_consist"]=consistent_artist
check["overall_consist"]=consistent_overall

# Comment Check Consistency Hard

In [155]:
claude_embed = np.load(f"clip_embeddings_claude.npy",allow_pickle=True)

In [156]:
gemini_embed = np.load(f"clip_embeddings_gemini.npy",allow_pickle=True)

In [157]:
openai_embed = np.load(f"clip_embeddings_openai.npy",allow_pickle=True)

In [158]:
embed_consistent_creative=[]
embed_consistent_artistic=[]
embed_consistent_overall=[]
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
for i in range(openai_embed.shape[0]):
    artistic_sim= cosine_similarity([claude_embed[i][0],gemini_embed[i][0],openai_embed[i][0]])
    if (artistic_sim > 0.70).all():
        artistic_con=1
        embed_consistent_artistic.append(1)
    else:
        artistic_con=0
        embed_consistent_artistic.append(0)
    creative_sim= cosine_similarity([claude_embed[i][1],gemini_embed[i][1],openai_embed[i][1]])
    if (creative_sim > 0.70).all():
        creative=1
        embed_consistent_creative.append(1)
    else:
        creative=0
        embed_consistent_creative.append(0)

    
    if artistic_con+creative==2:
        embed_consistent_overall.append(1)
    else:
        embed_consistent_overall.append(0)
    if i%2500==0:
        print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Currently at {i}")

2025-11-28 08:04:09: Start
2025-11-28 08:04:09: Currently at 0


In [159]:
np.sum(embed_consistent_overall)

np.int64(20)

In [165]:
np.sum(embed_consistent_creative)

np.int64(31)

In [160]:
check["embed_artistic_consist"]=embed_consistent_artistic
check["embed_creative_consist"]=embed_consistent_creative
check["embed_overall_consist"]=embed_consistent_overall